In [4]:
# Importing the packages

import pandas as pd
import numpy as np

from sklearn.naive_bayes import GaussianNB,CategoricalNB, BernoulliNB
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.impute import KNNImputer
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split

In [5]:
# Read the data

In [7]:
df = pd.read_csv("C:/projects/oct_python/data/input_files/bankloans.csv")

In [9]:
df.isnull().sum()

age           0
ed            0
employ        0
address       0
income        0
debtinc       0
creddebt      0
othdebt       0
default     150
dtype: int64

In [15]:
fresh_applicants = df.loc[(df['default'].isna()==True)]
old_applicants = df.loc[(df['default'].isna()==False)]

In [16]:
old_applicants.shape

(700, 9)

In [17]:
# Cleaning, Outlier capping

In [18]:
def outlier(x):
    x_lower = x.quantile(0.05)
    x_upper = x.quantile(0.95)
    x = x.clip(lower = x_lower, upper = x_upper)
    return x


In [19]:
# Apply

In [21]:
old_applicants.isnull().sum()

age         0
ed          0
employ      0
address     0
income      0
debtinc     0
creddebt    0
othdebt     0
default     0
dtype: int64

In [22]:
# Split the data into train and test

In [23]:
old_applicants.columns

Index(['age', 'ed', 'employ', 'address', 'income', 'debtinc', 'creddebt',
       'othdebt', 'default'],
      dtype='object')

In [24]:
X = old_applicants[['age', 'ed', 'employ', 'address', 'income', 'debtinc', 'creddebt',
       'othdebt']]
y = old_applicants['default']

In [30]:
X_train,X_test,y_train, y_test = train_test_split(X,y,test_size=0.20, random_state = 123)

In [31]:
X_test.shape

(140, 8)

In [32]:
old_applicants.shape

(700, 9)

In [33]:
# Outlier capping on Train and test

In [35]:
X_train = X_train.apply(outlier)

In [36]:
X_test = X_test.apply(outlier)

C:\Users\ravul\AppData\Local\Temp\ipykernel_23696\2434008243.py:4: FutureWarning: Downcasting behavior in Series and DataFrame methods 'where', 'mask', and 'clip' is deprecated. In a future version this will not infer object dtypes or cast all-round floats to integers. Instead call result.infer_objects(copy=False) for object inference, or cast round floats explicitly. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  x = x.clip(lower = x_lower, upper = x_upper)


In [37]:
# Building the model

In [39]:
gnb = GaussianNB()

In [40]:
gnb.fit(X_train, y_train)

,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",1e-09


In [47]:
print(f"Accuracy score on train set {accuracy_score(y_train,gnb.predict(X_train))}")

Accuracy score on train set 0.7785714285714286


In [46]:
print(f"Accuracy score on test set {accuracy_score(y_test,gnb.predict(X_test))}")

Accuracy score on test set 0.8


In [48]:
# Rebuilding the model with VIF

In [51]:
old_applicants_X = old_applicants[['age', 'ed', 'employ', 'address', 'income', 'debtinc', 'creddebt',
       'othdebt']]

In [52]:
values = [variance_inflation_factor(old_applicants_X.values, i) for i in range((old_applicants_X.shape[1]))]
for i in range(old_applicants_X.shape[1]):
    df = pd.DataFrame(columns = ["Feature", "vif_value"]) # Table creation
    df["Feature"] = old_applicants_X.columns # Correct
    df["vif_value"] = values 

In [70]:
df

,Feature,vif_value
0,age,16.650978
1,ed,4.981880
2,employ,5.703036
3,address,3.531411
4,income,10.477181
5,debtinc,8.639826
6,creddebt,3.718221
7,othdebt,6.450959


In [71]:
sel_variables = df.loc[(df['vif_value'] < 10 ),'Feature'].to_list()

In [72]:
sel_variables

['ed', 'employ', 'address', 'debtinc', 'creddebt', 'othdebt']

In [73]:
gnb.fit(X_train[sel_variables], y_train)

,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",1e-09


In [74]:
print(f"Accuracy score on train set {accuracy_score(y_train,gnb.predict(X_train[sel_variables]))}")

Accuracy score on train set 0.775


In [75]:
print(f"Accuracy score on train set {accuracy_score(y_test,gnb.predict(X_test[sel_variables]))}")

Accuracy score on train set 0.7571428571428571


In [77]:
from sklearn.metrics import classification_report

In [79]:
print(classification_report(y_train,gnb.predict(X_train[sel_variables])))

              precision    recall  f1-score   support

         0.0       0.83      0.87      0.85       413
         1.0       0.58      0.50      0.54       147

    accuracy                           0.78       560
   macro avg       0.71      0.69      0.69       560
weighted avg       0.77      0.78      0.77       560



In [80]:
print(classification_report(y_test,gnb.predict(X_test[sel_variables])))

              precision    recall  f1-score   support

         0.0       0.81      0.88      0.84       104
         1.0       0.54      0.39      0.45        36

    accuracy                           0.76       140
   macro avg       0.67      0.64      0.65       140
weighted avg       0.74      0.76      0.74       140



In [81]:
y_train.value_counts()

default
0.0    413
1.0    147
Name: count, dtype: int64

In [82]:
# Problem : Class Imbalance : techniques

In [83]:
# Classification Dataset

# Oversampling
# Undersampling
# SMOTE

In [85]:
sel_variables

['ed', 'employ', 'address', 'debtinc', 'creddebt', 'othdebt']

In [87]:
data = fresh_applicants[sel_variables]

In [88]:
data = data.apply(outlier)

In [90]:
data['Default'] = gnb.predict(data)

In [91]:
data.to_csv("C:/projects/oct_python/data/output_files/nb_outputs.csv", index = False)

In [92]:
# End of the notebook